# Preparation du corpus TEI
Preparation du corpus tei pour faire l'entrainement de la meme facon que pour les autres 
fine tunings.

In [1]:
from xml.sax import parseString
from xml.sax.handler import ContentHandler

import requests as rq
import pandas as pd

TAGS = ['O', 'B-PERS', 'I-PERS', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE' ]

class TeiToNer(ContentHandler):
    '''
    Cette classe est un handler pour traiter les documents TEI avec un parser SAX.
    
    Cette classe sert à récuperer une version processable d'un corpus annoté en TEI pour
    en faire un dataframe qu'on puisse facilement exploiter pour entrainer un modèle qui
    fasse du NER avec un encodage des entités au format IOB (inside, outside, begin).
    '''
    _text:   bool = False  # est-on dans un tag 'text' (je ne veux pas les headers)
    _div:    bool = False  # est-on dans un 'div' (ca délimite bien les datapoints)
    _choice: bool = False  # est-on dans un 'choice' (je ne veux que l'alternative 'orig')
    _orig:   bool = False  # est-on justement dans l'alternative orig
    _subst:  bool = False  # est-on dans un 'subst' (substitution de texte dans l'original)
    _add:    bool = False  # est-on dans la partie 'add' (cad ce qui doit etre lu dans l'original)
    _del:    bool = False  # est-on dans un 'del' (texte rayé)
    _begin:  bool = False  # est-on au début d'une entité
    _netag:  str  = 'O'    # quel est le tag ner courant ?

    _data:   list[dict]    # la liste de dict qu'on va transformer en dataframe qd on aura fini
    _curr:   list[str]     # la liste des tokens (à la grosse louche) composant le datapoint courant
    _labels: list[str]     # la liste des labels (ne tag) associé à chacun des tokens du datapoint
    
    def __init__(self):
        super().__init__()
        self._data   = []
        self._curr   = []
        self._labels = []
        
    def startElement(self, name, attrs):
        match name:
            case 'text':
                self._nextbloc()
                self._text = True
            case 'div' if self._text:
                self._nextbloc()
                self._div = True
            case 'choice' if self._div:
                self._choice = True
            case 'orig'|'abbr' if self._choice:
                self._orig = True
            case 'subst':
                self._subst = True
            case 'add': 
                self._add = True
            case 'del':
                self._del = True
            case 'date':
                self._begin = True
                self._netag = 'DATE'
            case 'placeName':
                self._begin = True
                self._netag = 'LOC'
            case 'persName': 
                self._begin = True
                self._netag = 'PERS'
                
    def endElement(self, name):
        match name:
            case 'text':
                self._text = False
                self._nextbloc()
            case 'div' if self._text:
                self._div = False
                self._nextbloc()
            case 'choice' if self._div:
                self._choice = False
            case 'orig'|'abbr' if self._choice:
                self._orig = False
            case 'subst':
                self._subst = False
            case 'add':
                self._add = False
            case 'del':
                self._del = False
            case 'date' | 'placeName' | 'persName':
                self._begin = False
                self._netag = 'O'
            
    def characters(self, content):
        if self._include_text():
            words = content.strip().split()
            labels= ([self._netag] * len(words) if self._netag == 'O' 
                     else [f'B-{self._netag}' if i == 0 and self._begin 
                     else  f'I-{self._netag}' for i in range(len(words))])
            self._begin = False
            self._curr.extend(words)
            self._labels.extend(labels)

    def _include_text(self):
        include = self._div
        include &= (not self._choice or self._orig)
        include &= (not self._subst or self._add)
        include &= not self._del
        return include
    
    def _nextbloc(self) -> pd.DataFrame:
        if self._curr:
            entry = {
                'text'  : self._curr, 
                'labels': self._labels }
            self._data.append(entry)
            self._curr   = []
            self._labels = []

    def to_df(self):
        return pd.DataFrame(self._data)



def data(url: str) -> pd.DataFrame:
    '''
    Renvoie un dataframe avec les informations contenues dans le texte décrit en TEI
    à l'adresse donnée
    '''
    rsp = rq.get(url)
    text= rsp.content # bytes actually (parser should process the encoding directive)
    hdlr= TeiToNer()
    parseString(text, hdlr)
    return hdlr.to_df()


In [2]:
import requests as rq
from bs4 import BeautifulSoup

def tei_resource(url: str) -> str:
    '''Cette fonction sert uniquement à retrouver l'url de la resource TEI sans avoir a scrapper plus que nécessaire'''
    return url.replace('view?docId=', 'data/').replace('&doc.view=notice;', '')

def documents(url: str, res: list[str] = []) -> list[str]:
    '''Cette fonction renvoie la liste des urls de tous les documents TEI qu'on pourrait trouver avec une recherche'''
    # process one page
    rsp = rq.get(url)
    text= rsp.content
    html= BeautifulSoup(text, 'html.parser')
    
    for doc in html.select("div.docHit"):
        row, *_ = doc.select('tr')
        link,*_ = row.select('td.col3 a[href]')
        res.append(link['href'])

    # extract links to the next pages of the search
    suivants = [tag for tag in html.select('div.resultsHeader td.right span.right a[href]') if tag.text == 'Suivant']
    if suivants:
        return documents(suivants[0]['href'], res)
    else:
        return [tei_resource(doc) for doc in res]

url = 'http://xtf.bvh.univ-tours.fr/xtf/search?type=tei;f1-subject=Archives'
docs= documents(url)
docs

['http://xtf.bvh.univ-tours.fr/xtf/data/tei/FRAD0330635101_1B298_104/FRAD0330635101_1B298_104_tei.xml',
 'http://xtf.bvh.univ-tours.fr/xtf/data/tei/FRAD0330635101_1B295_193/FRAD0330635101_1B295_193_tei.xml',
 'http://xtf.bvh.univ-tours.fr/xtf/data/tei/FRAD0330635101_1B298_188/FRAD0330635101_1B298_188_tei.xml',
 'http://xtf.bvh.univ-tours.fr/xtf/data/tei/FRAD0330635101_1B305_228/FRAD0330635101_1B305_228_tei.xml',
 'http://xtf.bvh.univ-tours.fr/xtf/data/tei/FRAD0330635101_1B280_118/FRAD0330635101_1B280_118_tei.xml',
 'http://xtf.bvh.univ-tours.fr/xtf/data/tei/FRAD0330635101_1B259_105/FRAD0330635101_1B259_105_tei.xml',
 'http://xtf.bvh.univ-tours.fr/xtf/data/tei/FRAD0330635101_1B258_149/FRAD0330635101_1B258_149_tei.xml',
 'http://xtf.bvh.univ-tours.fr/xtf/data/tei/FRAD0330635101_1B294_393/FRAD0330635101_1B294_393_tei.xml',
 'http://xtf.bvh.univ-tours.fr/xtf/data/tei/FRAD0330635101_1B269_163/FRAD0330635101_1B269_163_tei.xml',
 'http://xtf.bvh.univ-tours.fr/xtf/data/tei/FRAD0330635101_1B252

In [3]:
import pandas as pd

corpus = pd.concat((data(link) for link in docs), ignore_index=True)
corpus.index.name = 'sample'
corpus

,text,labels
sample,,
0,"[[Arrêt, du, Parlement, de, Bordeaux,, 10, déc...","[O, O, O, O, O, O, O, O, O, B-PERS, I-PERS, O,..."
1,"[[Arrêt, du, Parlement, de, Bordeaux,, 13, aoû...","[O, O, O, O, O, O, O, O, O, B-PERS, I-PERS, O,..."
2,"[[Arrêt, du, Parlement, de, Bordeaux,, 13, déc...","[O, O, O, O, O, O, O, O, O, B-PERS, I-PERS, I-..."
3,"[[Arrêt, du, Parlement, de, Bordeaux,, 14, aoû...","[O, O, O, O, O, O, O, O, O, B-PERS, I-PERS, I-..."
4,"[[Arrêt, du, Parlement, de, Bordeaux,, 14, avr...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-PERS..."
...,...,...
1220,"[Berry, Le, huictiesme, jour, d, aoust, oudict...","[B-LOC, O, B-DATE, I-DATE, I-DATE, I-DATE, I-D..."
1221,"[Don, Le, dernier, jour, dudict, mois, depesch...","[O, O, B-DATE, I-DATE, I-DATE, I-DATE, O, O, O..."
1222,"[Berry, Le, dernier, jour, d, aoust, mv, c, xl...","[B-LOC, O, B-DATE, I-DATE, I-DATE, I-DATE, I-D..."


In [5]:
corpus.columns = ['tokens', 'labels']
corpus.index.name = 'id'
corpus

,tokens,labels
id,,
0,"[[Arrêt, du, Parlement, de, Bordeaux,, 10, déc...","[O, O, O, O, O, O, O, O, O, B-PERS, I-PERS, O,..."
1,"[[Arrêt, du, Parlement, de, Bordeaux,, 13, aoû...","[O, O, O, O, O, O, O, O, O, B-PERS, I-PERS, O,..."
2,"[[Arrêt, du, Parlement, de, Bordeaux,, 13, déc...","[O, O, O, O, O, O, O, O, O, B-PERS, I-PERS, I-..."
3,"[[Arrêt, du, Parlement, de, Bordeaux,, 14, aoû...","[O, O, O, O, O, O, O, O, O, B-PERS, I-PERS, I-..."
4,"[[Arrêt, du, Parlement, de, Bordeaux,, 14, avr...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-PERS..."
...,...,...
1220,"[Berry, Le, huictiesme, jour, d, aoust, oudict...","[B-LOC, O, B-DATE, I-DATE, I-DATE, I-DATE, I-D..."
1221,"[Don, Le, dernier, jour, dudict, mois, depesch...","[O, O, B-DATE, I-DATE, I-DATE, I-DATE, O, O, O..."
1222,"[Berry, Le, dernier, jour, d, aoust, mv, c, xl...","[B-LOC, O, B-DATE, I-DATE, I-DATE, I-DATE, I-D..."


In [6]:
from datasets import Dataset, DatasetDict 

gold = Dataset.from_pandas(corpus)

dset = gold.train_test_split(train_size=.8, shuffle=True)
tv   = dset['test'].train_test_split(train_size=.5)

gold = DatasetDict({"train": dset["train"], "test": tv["train"], "valid": tv["test"]})
gold.save_to_disk('tei_corpus.datasetdict') # pas le droit de le charger sur hf hub
gold

/home/xgillard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Saving the dataset (1/1 shards): 100%|██████████████████████████████████████| 123/123 [00:00<00:00, 27237.18 examples/s]


DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 980
    })
    test: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 122
    })
    valid: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 123
    })
})